# API some country info  _______________

In [1]:
### imports
import requests

import matplotlib.pyplot as plt
import json
import pandas as pd

In [2]:
### what we wanna look up - GDP per capita, PPP (constant 2017 international $)
endpoint = 'https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.PP.KD?format=json&per_page=20000'

response = requests.get(endpoint)
r = response.json()

In [3]:
### make df - the first row is metadata not intended for df
data = r[1]
gdp_df = pd.json_normalize(data)

gdp_df

,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,AFE,2025,NaN,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZH,Africa Eastern and Southern
1,AFE,2024,4106.434140,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZH,Africa Eastern and Southern
2,AFE,2023,4083.993003,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZH,Africa Eastern and Southern
3,AFE,2022,4106.501907,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZH,Africa Eastern and Southern
4,AFE,2021,4056.113017,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZH,Africa Eastern and Southern
...,...,...,...,...,...,...,...,...,...,...
17551,ZWE,1964,NaN,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZW,Zimbabwe
17552,ZWE,1963,NaN,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZW,Zimbabwe
17553,ZWE,1962,NaN,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZW,Zimbabwe
17554,ZWE,1961,NaN,,,0,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",ZW,Zimbabwe


In [4]:
### what we wanna look up - Life expectancy at birth, total (years)
endpoint = 'https://api.worldbank.org/v2/country/all/indicator/SP.DYN.LE00.IN?format=json&per_page=20000'
response = requests.get(endpoint)
r = response.json()

### make df - the first row is metadata not intended for df
data = r[1]
life_df = pd.json_normalize(data)

life_df = life_df.dropna(subset=['value'])


life_df

,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
1,AFE,2024,65.349930,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZH,Africa Eastern and Southern
2,AFE,2023,65.146228,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZH,Africa Eastern and Southern
3,AFE,2022,64.487152,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZH,Africa Eastern and Southern
4,AFE,2021,62.979999,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZH,Africa Eastern and Southern
5,AFE,2020,63.766484,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZH,Africa Eastern and Southern
...,...,...,...,...,...,...,...,...,...,...
17551,ZWE,1964,55.431000,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZW,Zimbabwe
17552,ZWE,1963,54.942000,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZW,Zimbabwe
17553,ZWE,1962,54.453000,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZW,Zimbabwe
17554,ZWE,1961,53.966000,,,0,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",ZW,Zimbabwe


In [5]:
### format our life_df for merging
life_df.columns = life_df.columns.str.replace('.', '_')

life_df = life_df[[
    'country_value',
    'countryiso3code',
    'date',
    'value'
]]

life_df = life_df.rename(columns={
    'value': 'life_expectancy',
    'country_value': 'country'
})

life_df

,country,countryiso3code,date,life_expectancy
1,Africa Eastern and Southern,AFE,2024,65.349930
2,Africa Eastern and Southern,AFE,2023,65.146228
3,Africa Eastern and Southern,AFE,2022,64.487152
4,Africa Eastern and Southern,AFE,2021,62.979999
5,Africa Eastern and Southern,AFE,2020,63.766484
...,...,...,...,...
17551,Zimbabwe,ZWE,1964,55.431000
17552,Zimbabwe,ZWE,1963,54.942000
17553,Zimbabwe,ZWE,1962,54.453000
17554,Zimbabwe,ZWE,1961,53.966000


In [6]:
### format our gdp_df for merging
gdp_df.columns = gdp_df.columns.str.replace('.', '_')

gdp_df = gdp_df.dropna(subset=['value'])

gdp_df = gdp_df[[
    'country_value',
    'countryiso3code',
    'country_id',
    'date',
    'value'
]]

gdp_df = gdp_df.rename(columns={
    'value': 'gdp',
    'country_value': 'country'
})

gdp_df

,country,countryiso3code,country_id,date,gdp
1,Africa Eastern and Southern,AFE,ZH,2024,4106.434140
2,Africa Eastern and Southern,AFE,ZH,2023,4083.993003
3,Africa Eastern and Southern,AFE,ZH,2022,4106.501907
4,Africa Eastern and Southern,AFE,ZH,2021,4056.113017
5,Africa Eastern and Southern,AFE,ZH,2020,3978.931395
...,...,...,...,...,...
17521,Zimbabwe,ZWE,ZW,1994,6010.742417
17522,Zimbabwe,ZWE,ZW,1993,5509.083113
17523,Zimbabwe,ZWE,ZW,1992,5532.037403
17524,Zimbabwe,ZWE,ZW,1991,6254.274735


In [7]:
### merge and format the 2 df into gdp_life_df
gdp_life_df = life_df.merge(gdp_df, 'inner', ['countryiso3code', 'date'])
gdp_life_df = gdp_life_df.rename(columns={"country_x": "country"}).drop(columns=["country_y"])
gdp_life_df = gdp_life_df[['country', 'countryiso3code', 'country_id', 'date', 'gdp', 'life_expectancy']]
gdp_life_df

,country,countryiso3code,country_id,date,gdp,life_expectancy
0,Africa Eastern and Southern,AFE,ZH,2024,4106.434140,65.349930
1,Africa Eastern and Southern,AFE,ZH,2023,4083.993003,65.146228
2,Africa Eastern and Southern,AFE,ZH,2022,4106.501907,64.487152
3,Africa Eastern and Southern,AFE,ZH,2021,4056.113017,62.979999
4,Africa Eastern and Southern,AFE,ZH,2020,3978.931395,63.766484
...,...,...,...,...,...,...
8880,Zimbabwe,ZWE,ZW,1994,6010.742417,52.537000
8881,Zimbabwe,ZWE,ZW,1993,5509.083113,53.976000
8882,Zimbabwe,ZWE,ZW,1992,5532.037403,55.602000
8883,Zimbabwe,ZWE,ZW,1991,6254.274735,57.037000


In [8]:
### what we wanna look up - all country info
endpoint = 'https://api.worldbank.org/v2/country?format=json&per_page=20000'

response = requests.get(endpoint)
r = response.json()

### make df - the first row is metadata not intended for df
data = r[1]
country_df = pd.json_normalize(data)

### looks like any row that doesnt have lat or lon is an aggregate/non-country
country_df = country_df[country_df['longitude'] != '']

country_df = country_df.rename(columns={"name": "country", "capitalCity": "capital_city", "iso2Code":"iso2code"})
country_df

,id,iso2code,country,capital_city,longitude,latitude,region.id,region.iso2code,region.value,adminregion.id,adminregion.iso2code,adminregion.value,incomeLevel.id,incomeLevel.iso2code,incomeLevel.value,lendingType.id,lendingType.iso2code,lendingType.value
0,ABW,AW,Aruba,Oranjestad,-70.0167,12.5167,LCN,ZJ,Latin America & Caribbean,,,,HIC,XD,High income,LNX,XX,Not classified
2,AFG,AF,Afghanistan,Kabul,69.1761,34.5228,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",MNA,XQ,"Middle East, North Africa, Afghanistan & Pakis...",LIC,XM,Low income,IDX,XI,IDA
5,AGO,AO,Angola,Luanda,13.242,-8.81155,SSF,ZG,Sub-Saharan Africa,SSA,ZF,Sub-Saharan Africa (excluding high income),LMC,XN,Lower middle income,IBD,XF,IBRD
6,ALB,AL,Albania,Tirane,19.8172,41.3317,ECS,Z7,Europe & Central Asia,ECA,7E,Europe & Central Asia (excluding high income),UMC,XT,Upper middle income,IBD,XF,IBRD
7,AND,AD,Andorra,Andorra la Vella,1.5218,42.5075,ECS,Z7,Europe & Central Asia,,,,HIC,XD,High income,LNX,XX,Not classified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,XKX,XK,Kosovo,Pristina,20.926,42.565,ECS,Z7,Europe & Central Asia,ECA,7E,Europe & Central Asia (excluding high income),UMC,XT,Upper middle income,IDX,XI,IDA
292,YEM,YE,"Yemen, Rep.",Sana'a,44.2075,15.352,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",MNA,XQ,"Middle East, North Africa, Afghanistan & Pakis...",LIC,XM,Low income,IDX,XI,IDA
293,ZAF,ZA,South Africa,Pretoria,28.1871,-25.746,SSF,ZG,Sub-Saharan Africa,SSA,ZF,Sub-Saharan Africa (excluding high income),UMC,XT,Upper middle income,IBD,XF,IBRD
294,ZMB,ZM,Zambia,Lusaka,28.2937,-15.3982,SSF,ZG,Sub-Saharan Africa,SSA,ZF,Sub-Saharan Africa (excluding high income),LMC,XN,Lower middle income,IDX,XI,IDA


In [9]:
### merge and format that thang
gdp_life_country_df = gdp_life_df.merge(country_df, 'inner', 'country')

gdp_life_country_df = gdp_life_country_df.rename(columns={"countryiso3code":"iso3code"}).drop(columns=['id', 'country_id'])
gdp_life_country_df = gdp_life_country_df[[
    "country",
    "date",
    "gdp",
    "life_expectancy",
    "capital_city",
    "incomeLevel.value",
    "adminregion.value",
    "iso2code",
    "iso3code",
    
]]
gdp_life_country_df

,country,date,gdp,life_expectancy,capital_city,incomeLevel.value,adminregion.value,iso2code,iso3code
0,Afghanistan,2023,1983.812620,66.035,Kabul,Low income,"Middle East, North Africa, Afghanistan & Pakis...",AF,AFG
1,Afghanistan,2022,1981.710168,65.617,Kabul,Low income,"Middle East, North Africa, Afghanistan & Pakis...",AF,AFG
2,Afghanistan,2021,2144.166570,60.417,Kabul,Low income,"Middle East, North Africa, Afghanistan & Pakis...",AF,AFG
3,Afghanistan,2020,2769.685745,61.454,Kabul,Low income,"Middle East, North Africa, Afghanistan & Pakis...",AF,AFG
4,Afghanistan,2019,2927.245144,62.941,Kabul,Low income,"Middle East, North Africa, Afghanistan & Pakis...",AF,AFG
...,...,...,...,...,...,...,...,...,...
6708,Zimbabwe,1994,6010.742417,52.537,Harare,Lower middle income,Sub-Saharan Africa (excluding high income),ZW,ZWE
6709,Zimbabwe,1993,5509.083113,53.976,Harare,Lower middle income,Sub-Saharan Africa (excluding high income),ZW,ZWE
6710,Zimbabwe,1992,5532.037403,55.602,Harare,Lower middle income,Sub-Saharan Africa (excluding high income),ZW,ZWE
6711,Zimbabwe,1991,6254.274735,57.037,Harare,Lower middle income,Sub-Saharan Africa (excluding high income),ZW,ZWE
